# InvoiceAI — Evaluation on SROIE

Measures how accurate the pipeline is on real receipts with known answers.

**Method:** choose the setup on the **train** split (40 receipts), then report the final numbers on the **test** split (100 receipts). We never tune on the test split, so the final numbers are honest.

**Setups compared:**
| Name | Prompt | OCR fallback |
|---|---|---|
| `v2` | v2 | off — round 1 baseline |
| `v2+fb` | v2 | on |
| `v3+fb` | v3 | on |

**Time on a T4:** ≈ 30 min for train + ≈ 40 min for test. Keep this tab open.
With `USE_DRIVE = True`, the round 1 `v2` run is reused from Google Drive (no need to run it again), and interrupted runs continue.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO_DIR = "/content/invoice-ai"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/invoice-ai.git {REPO_DIR}
!ls {REPO_DIR}/app {REPO_DIR}/eval

## 3. Install requirements

If Colab asks you to **restart the session**, click Restart and continue with step 4.

In [ ]:
!pip install -q -r /content/invoice-ai/requirements.txt

## 4. Where to save results

In [ ]:
USE_DRIVE = True  # recommended: reuses round 1 results and survives disconnects

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/invoice-ai-eval"
else:
    OUTPUT_ROOT = "/content/invoice-ai/eval"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("Results folder:", OUTPUT_ROOT)
print("Already there:", sorted(os.listdir(OUTPUT_ROOT)))

## 5. Load the models

In [ ]:
import os, sys, json, time, logging
os.chdir("/content/invoice-ai")
sys.path.insert(0, "/content/invoice-ai")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", stream=sys.stdout, force=True)
logging.getLogger("app").setLevel(logging.WARNING)  # short output during evaluation

from app.pipeline import get_pipeline
from eval.evaluate import load_sroie, run_evaluation, worst_examples

pipeline = get_pipeline()
start = time.time()
pipeline.load()
print(f"Models loaded in {time.time() - start:.1f} s")

## 6. Check the dataset format

Always look at the data before trusting a number.

In [ ]:
from datasets import load_dataset
from IPython.display import display

ds = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")
print(ds)
example = ds[0]
print("\nKey:", example["key"])
print("Labels:", json.dumps(example["entities"], indent=2))
preview = example["image"].copy()
preview.thumbnail((400, 800))
display(preview)

## 7. Quick test on 3 receipts

`ok` = correct, `x` = wrong, `-` = no label.

In [ ]:
pipeline.use_ocr_fallback = True
report = run_evaluation(pipeline, load_sroie("train", limit=3), output_dir=f"{OUTPUT_ROOT}/runs/smoke_test",
                        prompt_version="v3", split="train", resume=False)
print(json.dumps(report["summary"]["fields"], indent=1))

## 8. Compare the setups (train split, 40 receipts each)

`v2` is the round 1 baseline. With Google Drive it is loaded from the saved run (a few seconds).

In [ ]:
import pandas as pd

N_COMPARE = 40
SETUPS = {            # name: (prompt, OCR fallback on?, folder)
    "v2":    ("v2", False, "train_v2"),       # round 1 folder, reused
    "v2+fb": ("v2", True,  "train_v2_fb"),
    "v3+fb": ("v3", True,  "train_v3_fb"),
}

comparison = {}
for name, (prompt, use_fallback, folder) in SETUPS.items():
    print(f"\n===== {name} =====")
    pipeline.use_ocr_fallback = use_fallback
    report = run_evaluation(pipeline, load_sroie("train", limit=N_COMPARE), output_dir=f"{OUTPUT_ROOT}/runs/{folder}",
                            prompt_version=prompt, split="train")
    comparison[name] = report["summary"]

rows = []
for field in comparison["v2"]["fields"]:
    row = {"field": field}
    for name, summary in comparison.items():
        row[name] = summary["fields"][field]["exact_match"]
    rows.append(row)
table = pd.DataFrame(rows)
display(table.style.format({c: "{:.1%}" for c in table.columns if c != "field"}))

def average_exact(summary):
    values = [m["exact_match"] for m in summary["fields"].values() if m["exact_match"] is not None]
    return sum(values) / len(values)

for name, summary in comparison.items():
    print(f"{name:6} average exact: {average_exact(summary):.1%}   {summary['avg_seconds_per_doc']} s/doc   fallback: {summary.get('fallback', {})}")

BEST = max(comparison, key=lambda n: average_exact(comparison[n]))
print("\nBest setup on train:", BEST)

## 9. Look at the remaining mistakes (best setup, train)

In [ ]:
best_folder = SETUPS[BEST][2]
for field in ["company", "date", "address", "total"]:
    print(f"\n----- {field} ({BEST}) -----")
    for ex in worst_examples(f"{OUTPUT_ROOT}/runs/{best_folder}", field, n=5):
        print(f"  label:     {ex['label']}")
        print(f"  predicted: {ex['predicted']}   (fuzzy {ex['fuzzy']}, confidence {ex['confidence']})")
        print()

## 10. Final evaluation (test split, 100 receipts, best setup)

≈ 40 minutes. These are the numbers for your README and Upwork.

In [ ]:
prompt, use_fallback, _ = SETUPS[BEST]
pipeline.use_ocr_fallback = use_fallback
FINAL_DIR = f"{OUTPUT_ROOT}/test_{BEST.replace('+', '_')}"
final = run_evaluation(pipeline, load_sroie("test", limit=100), output_dir=FINAL_DIR,
                       prompt_version=prompt, split="test")

## 11. Results: round 1 vs round 2 (test split)

In [ ]:
from IPython.display import Markdown

round1_path = f"{OUTPUT_ROOT}/results.json"   # round 1 final run (v2, no fallback)
if os.path.exists(round1_path):
    round1 = json.load(open(round1_path))["summary"]["fields"]
    round2 = final["summary"]["fields"]
    rows = [{"field": f, "round 1 (v2)": round1[f]["exact_match"], f"round 2 ({BEST})": round2[f]["exact_match"]} for f in round2]
    before_after = pd.DataFrame(rows)
    display(before_after.style.format({c: "{:.1%}" for c in before_after.columns if c != "field"}))
else:
    print("Round 1 results not found in", OUTPUT_ROOT, "- showing round 2 only.")

display(Markdown(open(f"{FINAL_DIR}/results.md").read()))

## 12. Download the result files

Upload `results.md` and `results.json` into the `eval/` folder on GitHub (they replace the round 1 files).

In [ ]:
from google.colab import files
files.download(f"{FINAL_DIR}/results.md")
files.download(f"{FINAL_DIR}/results.json")